In [1]:
import pandas as pd
from rdkit import Chem
import torch.nn.functional as F
from rdkit.Chem import AllChem
from rdkit.Chem import rdFingerprintGenerator, MolStandardize
from skfp.fingerprints import AtomPairFingerprint
from rdkit import RDLogger
from sklearn.metrics import f1_score, hamming_loss
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from transformers import Trainer, TrainingArguments
from sklearn.multioutput import MultiOutputClassifier
import lightgbm as lgb
import obonet
from torch_geometric.data import Data
from torch.utils.data import Dataset
import torch
from torch_geometric.loader import DataLoader
import torch.nn as nn
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from torch_geometric.nn import GCNConv, global_mean_pool
from transformers import AutoTokenizer, AutoModel
from transformers import AutoModelForSequenceClassification

In [2]:
test_df = pd.read_parquet("/kaggle/input/datasets/maciejwieteska/task1hackathon/chebi_dataset_test_empty.parquet")

print(test_df.head())

      mol_id                                             SMILES
0   mol_6861  OC[C@H]1O[C@H](O[C@H]2C(O)O[C@H](CO)[C@@H](O)[...
1  mol_29793  O.O=C([O-])CC(O)(CC(=O)[O-])C(=O)[O-].[K+].[K+...
2  mol_26953  CC(C)=CCC/C(C)=C/CC/C(C)=C/CC/C(C)=C\CC/C(C)=C...
3  mol_26053                                  C=C1CCC(C(C)C)CC1
4  mol_18653                                   O=C(O)C1CCCCC1=O


In [3]:
test_sample_df = pd.read_parquet("/kaggle/input/datasets/maciejwieteska/task1hackathon/chebi_submission_example.parquet")

print(test_sample_df.head())

      mol_id                                             SMILES  class_0  \
0   mol_6861  OC[C@H]1O[C@H](O[C@H]2C(O)O[C@H](CO)[C@@H](O)[...        0   
1  mol_29793  O.O=C([O-])CC(O)(CC(=O)[O-])C(=O)[O-].[K+].[K+...        1   
2  mol_26953  CC(C)=CCC/C(C)=C/CC/C(C)=C/CC/C(C)=C\CC/C(C)=C...        1   
3  mol_26053                                  C=C1CCC(C(C)C)CC1        1   
4  mol_18653                                   O=C(O)C1CCCCC1=O        0   

   class_1  class_2  class_3  class_4  class_5  class_6  class_7  ...  \
0        0        1        0        0        0        0        1  ...   
1        1        1        0        0        0        0        1  ...   
2        0        0        1        1        0        1        1  ...   
3        1        0        1        1        0        0        0  ...   
4        0        0        0        1        0        0        1  ...   

   class_490  class_491  class_492  class_493  class_494  class_495  \
0          1          1          

In [4]:
train_df = pd.read_parquet("/kaggle/input/datasets/maciejwieteska/task1hackathon/chebi_dataset_train.parquet")
print(train_df.head())


      mol_id                                             SMILES  class_0  \
0  mol_12500                            CCCCC/C=C\CCCCCCCC(=O)O        1   
1  mol_15962                           Cc1cc2cc(O)cc(O)c2c(C)n1        1   
2  mol_42147  C[C@H](CCC[C@@H](C)C=O)[C@H]1CC[C@H]2[C@@H]3CC...        1   
3  mol_43459  CCN=C1C=CC(=C(c2ccc(NCC)cc2)c2ccc(NCC)c(C)c2)C=C1        1   
4  mol_12734  C[C@H](CCC(O)=N[C@@H](Cc1c[nH]c2ccccc12)C(=O)O...        1   

   class_1  class_2  class_3  class_4  class_5  class_6  class_7  ...  \
0        1        1        1        1        1        1        1  ...   
1        1        1        1        1        1        1        1  ...   
2        1        1        1        1        1        1        1  ...   
3        1        1        1        1        1        1        0  ...   
4        1        1        1        1        1        1        1  ...   

   class_490  class_491  class_492  class_493  class_494  class_495  \
0          0          0          

In [5]:
class_cols = [c for c in train_df.columns if c.startswith("class_")]
y = train_df[class_cols] 
print(y.head())

   class_0  class_1  class_2  class_3  class_4  class_5  class_6  class_7  \
0        1        1        1        1        1        1        1        1   
1        1        1        1        1        1        1        1        1   
2        1        1        1        1        1        1        1        1   
3        1        1        1        1        1        1        1        0   
4        1        1        1        1        1        1        1        1   

   class_8  class_9  ...  class_490  class_491  class_492  class_493  \
0        1        1  ...          0          0          0          0   
1        1        1  ...          0          0          0          0   
2        1        1  ...          0          0          0          0   
3        0        1  ...          0          0          0          0   
4        1        1  ...          0          0          0          0   

   class_494  class_495  class_496  class_497  class_498  class_499  
0          0          0          0

In [6]:
afp_generator = AtomPairFingerprint(n_jobs=-1, sparse=False)

smiles_data = train_df["SMILES"] 

X_train = afp_generator.transform(smiles_data)

print(f"Kształt macierzy cech (X): {X_train.shape}")
print(f"Kształt macierzy etykiet (y): {y.shape}")

[07:23:13] WARNING: not removing hydrogen atom without neighbors
[07:23:13] WARNING: not removing hydrogen atom without neighbors
[07:23:13] WARNING: not removing hydrogen atom without neighbors
[07:23:13] WARNING: not removing hydrogen atom without neighbors
[07:23:13] WARNING: not removing hydrogen atom without neighbors
[07:23:13] WARNING: not removing hydrogen atom without neighbors
[07:23:13] WARNING: not removing hydrogen atom without neighbors
[07:23:13] WARNING: not removing hydrogen atom without neighbors
[07:23:13] WARNING: not removing hydrogen atom without neighbors
[07:23:13] WARNING: not removing hydrogen atom without neighbors
[07:23:13] WARNING: not removing hydrogen atom without neighbors
[07:23:13] WARNING: not removing hydrogen atom without neighbors
[07:23:13] WARNING: not removing hydrogen atom without neighbors
[07:23:13] WARNING: not removing hydrogen atom without neighbors
[07:23:13] WARNING: not removing hydrogen atom without neighbors
[07:23:14] WARNING: not r

Kształt macierzy cech (X): (33668, 2048)
Kształt macierzy etykiet (y): (33668, 500)


In [7]:
graph = obonet.read_obo("/kaggle/input/datasets/maciejwieteska/task1hackathon/chebi_classes.obo")

print(f"Liczba klas w grafie: {len(graph)}")

for node_id, data in list(graph.nodes(data=True))[:5]:
    print(node_id, data.get("name"))

Liczba klas w grafie: 500
class_0 chemical entity
class_1 molecular entity
class_10 heteroatomic molecular entity
class_100 oligosaccharide derivative
class_101 glycoside


In [8]:
class2idx = {c: i for i, c in enumerate(y.columns)}

edges = []

for child_id in graph.nodes():
    child_idx = class2idx.get(child_id)
    if child_idx is None:
        continue
    
    for parent_id in graph.successors(child_id):
        parent_idx = class2idx.get(parent_id)
        if parent_idx is not None:
            edges.append((child_idx, parent_idx))

print(f"Liczba krawędzi w DAG: {len(edges)}")

Liczba krawędzi w DAG: 748


In [9]:
def apply_hierarchy(pred, edges):
    pred_corrected = pred.copy()
    changed = True
    while changed:
        changed = False
        for child, parent in edges:
            mask = pred_corrected[:, child] > pred_corrected[:, parent]
            if mask.any():
                pred_corrected[mask, parent] = 1
                changed = True
    return pred_corrected

In [10]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y.values,
    test_size=0.2,
    random_state=42
)

In [31]:
def smiles_to_graph(smiles, y_label, fingerprint):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    
    node_feats = []
    for atom in mol.GetAtoms():
        node_feats.append([
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetFormalCharge(),
            atom.GetHybridization().numerator,
            atom.GetIsAromatic() * 1.0,       
            atom.GetTotalNumHs(),          
            atom.GetMass() * 0.01             
        ])
    x = torch.tensor(node_feats, dtype=torch.float)
    
    edge_indices = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_indices += [[i, j], [j, i]]
    edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
    
    y = torch.tensor(y_label, dtype=torch.float).view(1, -1)
    fp = torch.tensor(fingerprint, dtype=torch.float).view(1, -1)
    
    return Data(x=x, edge_index=edge_index, y=y, fingerprint=fp)

data_list = []
fps = afp_generator.transform(train_df["SMILES"]) 

for i, row in train_df.iterrows():
    g = smiles_to_graph(row["SMILES"], y.iloc[i].values, fps[i])
    if g: data_list.append(g)

train_data, val_data = train_test_split(data_list, test_size=0.2, random_state=42)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)

[08:25:41] WARNING: not removing hydrogen atom without neighbors
[08:25:41] WARNING: not removing hydrogen atom without neighbors
[08:25:41] WARNING: not removing hydrogen atom without neighbors
[08:25:41] WARNING: not removing hydrogen atom without neighbors
[08:25:41] WARNING: not removing hydrogen atom without neighbors
[08:25:41] WARNING: not removing hydrogen atom without neighbors
[08:25:41] WARNING: not removing hydrogen atom without neighbors
[08:25:42] WARNING: not removing hydrogen atom without neighbors
[08:25:42] WARNING: not removing hydrogen atom without neighbors
[08:25:42] WARNING: not removing hydrogen atom without neighbors
[08:25:42] WARNING: not removing hydrogen atom without neighbors
[08:25:42] WARNING: not removing hydrogen atom without neighbors
[08:25:42] WARNING: not removing hydrogen atom without neighbors
[08:25:42] WARNING: not removing hydrogen atom without neighbors
[08:25:42] WARNING: not removing hydrogen atom without neighbors
[08:25:42] WARNING: not r

In [57]:
from torch_geometric.nn import GCNConv, GlobalAttention
import torch.nn.functional as F
import torch.nn as nn

class ChemicalHybridNet(nn.Module):
    def __init__(self, node_in_channels, fp_dim, num_classes):
        super().__init__()
        self.conv1 = GCNConv(node_in_channels, 64)
        self.conv2 = GCNConv(64, 128)
        self.att_gate = nn.Sequential(nn.Linear(128, 1), nn.Sigmoid())
        self.pool = GlobalAttention(gate_nn=self.att_gate)
        self.fc = nn.Sequential(
            nn.Linear(128 + fp_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, data):
        x, edge_index, batch, fp = data.x, data.edge_index, data.batch, data.fingerprint
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = self.pool(x, batch)
        fp = fp.view(x.size(0), -1)
        combined = torch.cat([x, fp], dim=1)
        return self.fc(combined)

class FingerprintMLP(nn.Module):
    def __init__(self, fp_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(fp_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, data):
        fp = data.fingerprint.view(data.fingerprint.size(0), -1)
        return self.net(fp)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_gnn = ChemicalHybridNet(node_in_channels=7, fp_dim=X_train.shape[1], num_classes=500).to(device)
model_mlp = FingerprintMLP(fp_dim=X_train.shape[1], num_classes=500).to(device)

optimizer_gnn = torch.optim.Adam(model_gnn.parameters(), lr=0.001)
optimizer_mlp = torch.optim.Adam(model_mlp.parameters(), lr=0.001)

pos_counts = torch.tensor(y.values.sum(axis=0), dtype=torch.float)
neg_counts = len(y) - pos_counts
pos_weight = (neg_counts / (pos_counts + 1e-6)).clamp(max=100.0).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

scheduler_gnn = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_gnn, mode='max', factor=0.5, patience=5)
scheduler_mlp = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_mlp, mode='max', factor=0.5, patience=5)

/tmp/ipykernel_586/3301632076.py:12: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn=self.att_gate)


In [58]:
def train_step(model, optimizer, loader):
    model.train()
    total_loss = 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate_ensemble(loader, model_gnn, model_mlp, device, edges, gnn_weight=0.6):
    model_gnn.eval()
    model_mlp.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            logits_gnn = model_gnn(data)
            logits_mlp = model_mlp(data)
            
            avg_logits = (gnn_weight * logits_gnn) + ((1 - gnn_weight) * logits_mlp)
            preds = (torch.sigmoid(avg_logits) > 0.5).int().cpu().numpy()
            
            all_preds.append(preds)
            all_targets.append(data.y.cpu().numpy())
            
    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)
    y_pred_fixed = apply_hierarchy(y_pred, edges)
    
    mic = f1_score(y_true, y_pred_fixed, average='micro', zero_division=0)
    mac = f1_score(y_true, y_pred_fixed, average='macro', zero_division=0)
    return mic, mac

num_epochs = 120
best_gnn_macro = 0
best_mlp_macro = 0

for epoch in range(1, num_epochs + 1):
    loss_gnn = train_step(model_gnn, optimizer_gnn, train_loader)
    loss_mlp = train_step(model_mlp, optimizer_mlp, train_loader)
    
    _, mac_gnn = validate(val_loader, model_gnn, device, edges)
    _, mac_mlp = validate(val_loader, model_mlp, device, edges)
    
    ens_mic, ens_mac = validate_ensemble(val_loader, model_gnn, model_mlp, device, edges)
    
    scheduler_gnn.step(mac_gnn)
    scheduler_mlp.step(mac_mlp)
    
    print(f'Epoch: {epoch:02d} | GNN Loss: {loss_gnn:.4f} | MLP Loss: {loss_mlp:.4f}')
    print(f'    >> ENS F1 Micro: {ens_mic:.4f} | ENS F1 Macro: {ens_mac:.4f}')
    
    if mac_gnn > best_gnn_macro:
        best_gnn_macro = mac_gnn
        torch.save(model_gnn.state_dict(), 'best_gnn_model.pt')
        print("    [GNN model saved!]")
    
    if mac_mlp > best_mlp_macro:
        best_mlp_macro = mac_mlp
        torch.save(model_mlp.state_dict(), 'best_mlp_model.pt')
        print("    [MLP model saved!]")


print("\n--- Rozpoczynam końcową optymalizację progów na zbiorze walidacyjnym ---")

model_gnn.load_state_dict(torch.load('best_gnn_model.pt'))
model_mlp.load_state_dict(torch.load('best_mlp_model.pt'))

def optimize_thresholds(loader, model_gnn, model_mlp, device, num_classes=500):
    model_gnn.eval()
    model_mlp.eval()
    all_probs, all_targets = [], []
    
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            l_gnn, l_mlp = model_gnn(data), model_mlp(data)
            avg_probs = torch.sigmoid((0.6 * l_gnn) + (0.4 * l_mlp)).cpu().numpy()
            all_probs.append(avg_probs)
            all_targets.append(data.y.cpu().numpy())
            
    y_probs, y_true = np.vstack(all_probs), np.vstack(all_targets)
    best_thresholds = np.zeros(num_classes)
    
    for i in range(num_classes):
        if y_true[:, i].sum() > 0:
            thresholds = np.linspace(0.05, 0.8, 16)
            f1s = [f1_score(y_true[:, i], (y_probs[:, i] > t).astype(int), zero_division=0) for t in thresholds]
            best_thresholds[i] = thresholds[np.argmax(f1s)]
        else:
            best_thresholds[i] = 0.3
    return best_thresholds

best_thresholds = optimize_thresholds(val_loader, model_gnn, model_mlp, device)
print("Optymalizacja progów zakończona.")

Epoch: 01 | GNN Loss: 0.5067 | MLP Loss: 0.5395
    >> ENS F1 Micro: 0.4875 | ENS F1 Macro: 0.3024
    [GNN model saved!]
    [MLP model saved!]
Epoch: 02 | GNN Loss: 0.3038 | MLP Loss: 0.3719
    >> ENS F1 Micro: 0.5700 | ENS F1 Macro: 0.3640
    [GNN model saved!]
    [MLP model saved!]
Epoch: 03 | GNN Loss: 0.2372 | MLP Loss: 0.3260
    >> ENS F1 Micro: 0.6004 | ENS F1 Macro: 0.3992
    [GNN model saved!]
    [MLP model saved!]
Epoch: 04 | GNN Loss: 0.2004 | MLP Loss: 0.2965
    >> ENS F1 Micro: 0.6282 | ENS F1 Macro: 0.4267
    [GNN model saved!]
    [MLP model saved!]
Epoch: 05 | GNN Loss: 0.1753 | MLP Loss: 0.2769
    >> ENS F1 Micro: 0.6518 | ENS F1 Macro: 0.4462
    [GNN model saved!]
    [MLP model saved!]
Epoch: 06 | GNN Loss: 0.1576 | MLP Loss: 0.2624
    >> ENS F1 Micro: 0.6757 | ENS F1 Macro: 0.4764
    [GNN model saved!]
    [MLP model saved!]
Epoch: 07 | GNN Loss: 0.1470 | MLP Loss: 0.2493
    >> ENS F1 Micro: 0.6984 | ENS F1 Macro: 0.4949
    [GNN model saved!]
    [MLP

In [59]:
test_df = pd.read_parquet("/kaggle/input/datasets/maciejwieteska/task1hackathon/chebi_dataset_test_empty.parquet")

test_fps = afp_generator.transform(test_df["SMILES"])
test_data_list = []
for i, row in test_df.iterrows():
    dummy_y = np.zeros(500)
    g = smiles_to_graph(row["SMILES"], dummy_y, test_fps[i])
    if g:
        g.mol_id = row["mol_id"]
        test_data_list.append(g)

test_loader = DataLoader(test_data_list, batch_size=32, shuffle=False)

model_gnn_eval = ChemicalHybridNet(node_in_channels=7, fp_dim=test_fps.shape[1], num_classes=500).to(device)
model_mlp_eval = FingerprintMLP(fp_dim=test_fps.shape[1], num_classes=500).to(device)

model_gnn_eval.load_state_dict(torch.load('best_gnn_model.pt'))
model_mlp_eval.load_state_dict(torch.load('best_mlp_model.pt'))

model_gnn_eval.eval()
model_mlp_eval.eval()

all_preds = []
mol_ids = []
gnn_weight = 0.6 

print("Generowanie predykcji Ensemble...")
with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        
        logits_gnn = model_gnn_eval(data)
        logits_mlp = model_mlp_eval(data)
        
        avg_logits = (gnn_weight * logits_gnn) + ((1 - gnn_weight) * logits_mlp)
        probs = torch.sigmoid(avg_logits).cpu().numpy()
        preds = (probs > best_thresholds).astype(int)
        
        all_preds.append(preds)
        ids = data.mol_id
        mol_ids.extend(ids.tolist() if torch.is_tensor(ids) else ids)

y_pred_final = np.vstack(all_preds)
y_pred_corrected = apply_hierarchy(y_pred_final, edges)

class_cols = [c for c in train_df.columns if c.startswith("class_")]
id_to_smiles = dict(zip(test_df['mol_id'], test_df['SMILES']))

submission_df = pd.DataFrame(y_pred_corrected, columns=class_cols)
submission_df.insert(0, "mol_id", mol_ids)
submission_df.insert(1, "SMILES", submission_df["mol_id"].map(id_to_smiles))

submission_df.to_parquet("chebi_predictions_test.parquet", index=False)
print("Gotowe! Wyniki Ensemble zapisano.")

[10:00:23] WARNING: not removing hydrogen atom without neighbors
[10:00:23] WARNING: not removing hydrogen atom without neighbors
[10:00:23] WARNING: not removing hydrogen atom without neighbors
[10:00:23] WARNING: not removing hydrogen atom without neighbors
[10:00:23] WARNING: not removing hydrogen atom without neighbors
[10:00:23] WARNING: not removing hydrogen atom without neighbors
[10:00:23] WARNING: not removing hydrogen atom without neighbors
[10:00:23] WARNING: not removing hydrogen atom without neighbors
[10:00:23] WARNING: not removing hydrogen atom without neighbors
[10:00:23] WARNING: not removing hydrogen atom without neighbors
[10:00:23] WARNING: not removing hydrogen atom without neighbors
[10:00:23] WARNING: not removing hydrogen atom without neighbors
[10:00:23] WARNING: not removing hydrogen atom without neighbors
[10:00:23] WARNING: not removing hydrogen atom without neighbors
[10:00:23] WARNING: not removing hydrogen atom without neighbors
[10:00:23] WARNING: not r

Generowanie predykcji Ensemble...
Gotowe! Wyniki Ensemble zapisano.


In [67]:
from IPython.display import FileLink
checkpoint = {
    'gnn_state': model_gnn.state_dict(),
    'mlp_state': model_mlp.state_dict(),
    'thresholds': best_thresholds,
    'gnn_weight': 0.6 
}

torch.save(checkpoint, 'chemical_ensemble_final.pt')
FileLink('chemical_ensemble_final.pt')

/kaggle/working/chemical_ensemble_final.pt